# Practical 9 — Unsupervised Learning: k-Means Clustering and PCA

**Lectures:** 18. Clustering and K-Means · 19. Dimensionality Reduction

No labels this week. You will

1. implement **Lloyd's algorithm** for k-means, watch it converge, and see that it can get stuck in **local optima**;
2. implement **k-means++** initialisation and choose $k$ with the **elbow method** and silhouette score;
3. segment real **wholesale customers**;
4. implement **Principal Component Analysis** from scratch and use it to visualise 30-dimensional data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

## 1. Lloyd's algorithm (≈30 min)
Given $k$, choose initial centroids $\boldsymbol\mu_1,\dots,\boldsymbol\mu_k$, then repeat until the assignments stop changing:

1. **assignment step:** $c^{(i)} = \arg\min_j \lVert\mathbf x^{(i)} - \boldsymbol\mu_j\rVert^2$
2. **update step:** $\boldsymbol\mu_j = $ mean of the points assigned to cluster $j$

The objective being decreased is the **inertia** (within-cluster sum of squares) $\sum_i \lVert\mathbf x^{(i)}-\boldsymbol\mu_{c^{(i)}}\rVert^2$.

**TODO 1.1** Implement `kmeans` with **random initialisation** (pick $k$ distinct data points). Return the
centroids, the labels, the inertia and the list of centroid positions at every iteration.
If a cluster becomes empty, keep its previous centroid.

In [ ]:
X, y_true = make_blobs(n_samples=600, centers=5, cluster_std=1.0, random_state=3)
plt.scatter(*X.T, s=6); plt.title("unlabelled data"); plt.show()

def inertia(X, centroids, labels):
    return float(((X - centroids[labels]) ** 2).sum())

def assign(X, centroids):
    # YOUR CODE HERE
    raise NotImplementedError

def kmeans(X, k, init=None, max_iters=100, seed=0):
    g = np.random.default_rng(seed)
    centroids = X[g.choice(len(X), k, replace=False)] if init is None else init.copy()
    history = [centroids.copy()]
    labels = None
    for _ in range(max_iters):
        # YOUR CODE HERE
        raise NotImplementedError
    return centroids, labels, inertia(X, centroids, labels), history

C, lab, J, hist = kmeans(X, 5, seed=0)
print(f"converged after {len(hist) - 1} updates, inertia = {J:.1f}")

**TODO 1.2** Plot the final clusters (colour by label) and draw the path each centroid took across iterations.

In [ ]:
# YOUR CODE HERE

**TODO 1.3 — Local optima.** Run your k-means with $k=5$ from 20 different random seeds. Plot a histogram of the
final inertias and show the best and worst clusterings side by side. Why does this happen, and what is the simple fix?

In [ ]:
# YOUR CODE HERE

## 2. k-means++ and choosing k (≈25 min)
**k-means++** picks the first centroid uniformly at random, then each next centroid with probability proportional to
$D(\mathbf x)^2$, the squared distance from $\mathbf x$ to the nearest centroid chosen so far.

**TODO 2.1** Implement `kmeans_pp_init` and repeat the 20-seed experiment. Does the spread of inertias shrink?

In [ ]:
def kmeans_pp_init(X, k, seed=0):
    g = np.random.default_rng(seed)
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

**TODO 2.2 — Elbow method.** For $k = 1,\dots,10$ record the best inertia over 10 k-means++ runs, and plot inertia
against $k$. Also plot the **silhouette score** (`sklearn.metrics.silhouette_score`) for $k=2,\dots,10$.
Where is the "elbow"? Do the two methods agree? Finally, check that `sklearn.cluster.KMeans(n_clusters=5, n_init=10)`
reaches the same inertia as your implementation.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

# YOUR CODE HERE

## 3. Customer segmentation (≈20 min)
The **wholesale customers** data record annual spending (monetary units) of 440 clients of a distributor on six product
categories. `Channel` (1 = Hotel/Restaurant/Café, 2 = Retail) and `Region` are *not* used for clustering.

**TODO 3.1** Spending is heavily skewed, so apply `np.log1p` and then standardise the six spending columns. Use the elbow
method and silhouette to choose $k$ (try 2–8) with `KMeans(n_init=10, random_state=0)`. For your chosen $k$, show the
**mean original spending per cluster** (a table) and a cross-tabulation of cluster against `Channel`. Describe each segment
in one sentence, as you would to a marketing manager.

In [ ]:
customers = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/wholesale-customers.csv")
spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
print(customers[spend_cols].describe().round(0))
# YOUR CODE HERE

*Write your answer here.*

## 4. Principal Component Analysis from scratch (≈30 min)
PCA finds orthogonal directions of maximum variance. For centred data $X_c$ (each column mean 0):

1. covariance matrix $\Sigma = \frac{1}{m-1}X_c^\top X_c$;
2. eigen-decomposition $\Sigma\mathbf v_j = \lambda_j\mathbf v_j$, sorted so that $\lambda_1\ge\lambda_2\ge\dots$;
3. project onto the first $d$ eigenvectors: $Z = X_c V_d$. The **explained variance ratio** of component $j$ is $\lambda_j/\sum_i\lambda_i$.

**TODO 4.1** Implement `pca_fit` / `pca_transform` with `np.linalg.eigh` (for symmetric matrices). Apply it to the
standardised **breast-cancer** data (30 features). Check your explained variance ratios against
`sklearn.decomposition.PCA`, and your projections too — up to a **sign flip** per component (why?).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

bc = load_breast_cancer()
Xs = StandardScaler().fit_transform(bc.data)

def pca_fit(X, d):
    # YOUR CODE HERE
    raise NotImplementedError

def pca_transform(model, X):
    # YOUR CODE HERE
    raise NotImplementedError

model = pca_fit(Xs, 2)
Z = pca_transform(model, Xs)
sk = PCA(n_components=2).fit(Xs)
print("ours:   ", model["explained_variance_ratio"].round(4))
print("sklearn:", sk.explained_variance_ratio_.round(4))
print("projections equal up to sign:", np.allclose(np.abs(Z), np.abs(sk.transform(Xs))))

**TODO 4.2** (a) Plot the **cumulative** explained variance ratio for all 30 components; how many components are needed for
95 % of the variance? (b) Scatter-plot the data on the first two components, coloured by diagnosis (labels are used **only**
for colouring). (c) Which original features have the largest weights (loadings) in PC1?

In [ ]:
# YOUR CODE HERE

## 5. Extension: PCA + k-means, and the curse of dimensionality
* Run k-means ($k=2$) on the first 2 principal components of the breast-cancer data. How well do the clusters agree
  with the diagnoses (`sklearn.metrics.adjusted_rand_score`)? Compare with clustering all 30 standardised features.
* **Curse of dimensionality:** draw 500 uniform random points in $[0,1]^d$ for $d \in \{2, 10, 100, 1000\}$ and plot a
  histogram of all pairwise distances divided by their mean. What happens to the *contrast* between near and far points?
  What does that mean for k-NN and k-means in high dimensions?

In [ ]:
# YOUR CODE HERE

## Lab report checklist (Practical 9)
1. Your k-means clustering figure with centroid paths, and the local-optima experiment (histogram + best/worst), with random vs k-means++ statistics.
2. Elbow and silhouette plots for the blobs and for the customers; your chosen $k$ and justification.
3. The customer segment table and your one-sentence description of each segment.
4. PCA: explained-variance comparison with scikit-learn, the cumulative-variance plot, the 2-D projection and the top PC1 loadings.
5. A short reflection: what is fundamentally different about evaluating an *unsupervised* model compared with the supervised models of earlier weeks?